# 02. 학습과 비교 실험 (YOLOv8)

Roboflow에서 내보낸 데이터셋으로 여러 설정을 학습하고, **테스트 세트** 기준 성능과 속도를 한 표로 비교합니다.

**시작 전 확인**
1. 런타임 → 런타임 유형 변경 → **T4 GPU**
2. Roboflow 내보내기에서 **Resize를 끄거나 "Fit within 1280×1280"** 으로 설정했는지 (기본값 640이면 해상도 비교가 무의미해짐, `docs/labeling_guide.md` 참고)
3. Colab은 연결이 끊기면 파일이 사라지므로, 셀 1에서 Google Drive 저장을 켜 두는 것을 권장합니다.

In [ ]:
# ===== 셀 1: 환경 준비 =====
REPO_URL = "https://github.com/<YOUR_ID>/beeguard-hornet-tracker.git"   # 본인 저장소 주소로 바꾸기
USE_DRIVE = True   # 학습 결과를 Google Drive에 저장 (연결이 끊겨도 보존)

!pip -q install ultralytics
import os, subprocess
if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=False)
REPO = "/content/repo" if os.path.isdir("/content/repo") else "/content"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/beeguard/runs"
else:
    PROJECT = "/content/runs"
os.makedirs(PROJECT, exist_ok=True)
print("REPO:", REPO, "| 학습 결과 저장 위치:", PROJECT)

## 데이터셋 가져오기
둘 중 하나만 실행합니다.
- **방법 A**: Roboflow에서 YOLOv8 형식 zip을 내려받아 업로드
- **방법 B**: Roboflow의 "Download code"에 나오는 코드를 붙여넣기 (본인 프로젝트, 본인 API 키)

In [ ]:
# ===== 셀 2-A: zip 업로드 =====
from google.colab import files
import zipfile, shutil
up = files.upload()                      # Roboflow에서 받은 zip 선택
zip_name = next(iter(up))
shutil.rmtree("/content/dataset", ignore_errors=True)
zipfile.ZipFile(zip_name).extractall("/content/dataset")
print(os.listdir("/content/dataset"))

In [ ]:
# ===== 셀 2-B: Roboflow API (A 대신 사용) =====
# !pip -q install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="본인_API_키")
# project = rf.workspace("본인_워크스페이스").project("본인_프로젝트")
# ds = project.version(1).download("yolov8", location="/content/dataset")

In [ ]:
# ===== 셀 3: data.yaml 경로 정리 + 부류 확인 =====
import yaml
DS = "/content/dataset"
EXPECTED = ["giant_hornet", "honeybee", "yellow_legged_hornet"]

def fix_yaml(ds):
    p = os.path.join(ds, "data.yaml")
    d = yaml.safe_load(open(p))
    d["path"] = ds
    for key, sub in [("train", "train"), ("val", "valid"), ("test", "test")]:
        if os.path.isdir(os.path.join(ds, sub, "images")):
            d[key] = f"{sub}/images"
        elif key == "test":
            d.pop("test", None)
    yaml.safe_dump(d, open(p, "w"), allow_unicode=True, sort_keys=False)
    return p, d

DATA, cfg = fix_yaml(DS)
names = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][k] for k in sorted(cfg["names"])]
print("부류:", names)
assert names == EXPECTED, f"부류 이름/순서가 다릅니다. Roboflow 라벨 이름을 확인하세요: {names}"
HAS_TEST = "test" in cfg
for sub in ["train", "valid", "test"]:
    d = os.path.join(DS, sub, "images")
    if os.path.isdir(d):
        n_img = len(os.listdir(d))
        n_null = sum(1 for f in os.listdir(os.path.join(DS, sub, "labels"))
                     if os.path.getsize(os.path.join(DS, sub, "labels", f)) == 0)
        print(f"{sub:6s} 이미지 {n_img:4d}장 (박스 없는 이미지 {n_null}장)")

In [ ]:
# ===== 셀 4: 작은 객체 분석 — 입력 해상도별 객체 크기 =====
!python {REPO}/scripts/label_size_stats.py --root {DS} --imgsz 640 960 1280 \
    --names giant_hornet honeybee yellow_legged_hornet --csv {REPO}/results/label_size_stats.csv

## 비교 실험
`RUN`에 넣은 실험만 학습합니다. 핵심 3개(입력 크기 비교)를 먼저 돌리고, 시간이 남으면 나머지를 추가하세요.

| 이름 | 비교 목적 |
| --- | --- |
| n640 / n960 / n1280 | 입력 해상도가 작은 객체(꿀벌) 탐지에 주는 영향 |
| s960 | 모델 크기(n → s)의 영향 |
| n960_hue | 색상 증강을 세게(hsv_h 0.1) 주면 종 구분이 나빠지는지 (기본값 0.015) |
| p2_960 | 4픽셀 간격 격자(P2 헤드)를 추가하면 작은 객체가 나아지는지 |

In [ ]:
# ===== 셀 5: 실험 설정 =====
EPOCHS, PATIENCE, SEED = 100, 20, 0
EXPERIMENTS = {
    "n640":     dict(model="yolov8n.pt", imgsz=640),
    "n960":     dict(model="yolov8n.pt", imgsz=960),
    "n1280":    dict(model="yolov8n.pt", imgsz=1280),
    "s960":     dict(model="yolov8s.pt", imgsz=960),
    "n960_hue": dict(model="yolov8n.pt", imgsz=960, hsv_h=0.1),
    "p2_960":   dict(model="yolov8n-p2.yaml", weights="yolov8n.pt", imgsz=960),
}
RUN = ["n640", "n960", "n1280"]          # 먼저 돌릴 실험

In [ ]:
# ===== 셀 6: 학습 + 테스트 평가 =====
import json, pandas as pd
from ultralytics import YOLO

def train_and_eval(name, cfg):
    cfg = dict(cfg)
    model = YOLO(cfg.pop("model"))
    w = cfg.pop("weights", None)
    if w:
        model = model.load(w)                          # P2 구조에 사전학습 가중치 이식
    imgsz = cfg.pop("imgsz")
    model.train(data=DATA, epochs=EPOCHS, patience=PATIENCE, imgsz=imgsz, batch=-1, seed=SEED,
                project=PROJECT, name=name, exist_ok=True, plots=True, **cfg)
    best = os.path.join(PROJECT, name, "weights", "best.pt")
    m = YOLO(best).val(data=DATA, imgsz=imgsz, split="test" if HAS_TEST else "val",
                       project=PROJECT, name=f"{name}_test", exist_ok=True, plots=True)
    sp = m.speed                                       # 이미지 1장당 ms
    ms = sp["preprocess"] + sp["inference"] + sp["postprocess"]
    row = {"exp": name, "imgsz": imgsz, "precision": m.box.mp, "recall": m.box.mr,
           "mAP50": m.box.map50, "mAP50-95": m.box.map,
           "ms_per_img": ms, "fps": 1000 / ms if ms else float("nan"), "best_pt": best}
    for i, c in enumerate(m.box.ap_class_index):
        row[f"AP50_{names[int(c)]}"] = m.box.ap50[i]
    return row

rows = []
for name in RUN:
    print(f"\n===== {name} =====")
    rows.append(train_and_eval(name, EXPERIMENTS[name]))
    pd.DataFrame(rows).to_csv(os.path.join(PROJECT, "experiments.csv"), index=False)   # 중간 저장

df = pd.DataFrame(rows)
df.round(3)

In [ ]:
# ===== 셀 7: 가장 좋은 모델을 저장소 폴더로 정리 =====
import shutil
METRIC = "mAP50-95"                                  # 선택 기준
best_row = df.sort_values(METRIC, ascending=False).iloc[0]
print("선택:", best_row["exp"], f"({METRIC} {best_row[METRIC]:.3f})")

os.makedirs(f"{REPO}/models", exist_ok=True)
shutil.copy(best_row["best_pt"], f"{REPO}/models/best.pt")
df.drop(columns=["best_pt"]).round(4).to_csv(f"{REPO}/results/experiments.csv", index=False)

for exp in df["exp"]:
    out = f"{REPO}/results/{exp}"
    os.makedirs(out, exist_ok=True)
    for f in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "PR_curve.png"]:
        for src_dir in [os.path.join(PROJECT, exp), os.path.join(PROJECT, f"{exp}_test")]:
            src = os.path.join(src_dir, f)
            if os.path.exists(src):
                tag = "test_" if src_dir.endswith("_test") else ""
                shutil.copy(src, os.path.join(out, tag + f))
print(sorted(os.listdir(f"{REPO}/results")))

## 다음 단계
- `results/experiments.csv`의 표를 보고서 "학습 과정 및 성능"에 넣습니다. 정확도(mAP)와 속도(FPS)를 함께 적어야 Jetson 적용 판단 근거가 됩니다.
- 테스트 세트의 `test_confusion_matrix.png`에서 어떤 부류끼리 헷갈리는지 확인하고, 틀린 사진을 몇 장 골라 실패 사례로 분석합니다.
- 추적 영상은 `03_track.ipynb`에서 만듭니다.